In [ ]:
# ✅ NEW CHUNK 1: Pin NumPy to 1.x BEFORE installing anything else
# After this cell finishes, click "Restart Session" when prompted (or Runtime > Restart Session)
# Then continue from Chunk 2 onwards — do NOT re-run this cell after restart

!pip install -q "numpy<2.0.0"  # Pin to NumPy 1.x to avoid ABI mismatch
!pip install -q gptqmodel optimum accelerate datasets
!pip install -q "transformers>=4.45.0" --upgrade

# Verify numpy version
import numpy as np
print(f"NumPy version: {np.__version__}")
assert int(np.__version__.split(".")[0]) < 2, "❌ Still on NumPy 2.x — restart runtime now!"
print("✅ NumPy version OK. NOW restart the runtime before running any other chunk.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.34.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.34.1 which is incompatible.
NumPy version: 2.0.2


AssertionError: ❌ Still on NumPy 2.x — restart runtime now!

In [ ]:
# ✅ CHUNK 2: Verify GPU and environment (run after restart)

import torch
import numpy as np

print("=" * 50)
print(f"NumPy version   : {np.__version__}  ✅" if int(np.__version__.split(".")[0]) < 2 else f"NumPy version   : {np.__version__}  ⚠️ still 2.x!")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 50)

NumPy version   : 2.2.6  ⚠️ still 2.x!
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM total      : 15.64 GB


In [ ]:
# ✅ CHUNK 3: Load the base model (fp16) for baseline measurement

import torch
import time
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "facebook/opt-125m"

# ─── Tokenizer ──────────────────────────────────────────────────
print(f"Loading tokenizer for {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# ─── Model ──────────────────────────────────────────────────────
print(f"Loading base model in fp16 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,   # reduces CPU RAM spike during load
)
base_model.eval()

# ─── Size helper ────────────────────────────────────────────────
def get_model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)

base_size_mb = get_model_size_mb(base_model)
n_params     = sum(p.numel() for p in base_model.parameters())

print(f"\n✅ Base model loaded successfully")
print(f"   Parameters             : {n_params / 1e6:.1f}M")
print(f"   In-memory size (fp16)  : {base_size_mb:.1f} MB")
print(f"   Device                 : {next(base_model.parameters()).device}")

# Quick sanity inference test
test_input = tokenizer("Hello, my name is", return_tensors="pt").to("cuda")
with torch.no_grad():
    test_out = base_model.generate(**test_input, max_new_tokens=10)
print(f"   Sanity check output    : {tokenizer.decode(test_out[0], skip_special_tokens=True)}")
print("\n✅ Chunk 3 complete — proceed to Chunk 4")

Loading tokenizer for facebook/opt-125m ...


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


✅ Base model loaded successfully
   Parameters             : 125.2M
   In-memory size (fp16)  : 238.9 MB
   Device                 : cuda:0
   Sanity check output    : Hello, my name is J.C. and I am a student at

✅ Chunk 3 complete — proceed to Chunk 4


In [ ]:
# ✅ CHUNK 4: Baseline inference — timing + tokens/sec

PROMPTS = [
    "Artificial intelligence is transforming the world by",
    "The key to a successful machine learning model is",
    "Quantum computing will revolutionize cryptography because",
    "Climate change poses a significant threat to",
    "The history of the Roman Empire shows us that",
]

MAX_NEW_TOKENS = 50
WARMUP_RUNS = 2

def run_inference_benchmark(model, tokenizer, prompts, max_new_tokens=50, label="Model"):
    model.eval()
    timings = []
    token_counts = []
    outputs_text = []

    # Warmup
    for _ in range(WARMUP_RUNS):
        inp = tokenizer(prompts[0], return_tensors="pt").to("cuda")
        with torch.no_grad():
            model.generate(**inp, max_new_tokens=10)
    torch.cuda.synchronize()

    # Actual benchmark
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs["input_ids"].shape[1]

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
            )
        torch.cuda.synchronize()
        t1 = time.perf_counter()

        elapsed = t1 - t0
        new_tokens = out.shape[1] - input_len
        timings.append(elapsed)
        token_counts.append(new_tokens)
        outputs_text.append(tokenizer.decode(out[0], skip_special_tokens=True))

    avg_time   = np.mean(timings)
    avg_tokens = np.mean(token_counts)
    throughput = avg_tokens / avg_time

    print(f"\n{'='*55}")
    print(f"  {label} — Inference Benchmark")
    print(f"{'='*55}")
    print(f"  Avg latency per prompt : {avg_time*1000:.1f} ms")
    print(f"  Avg tokens generated   : {avg_tokens:.0f}")
    print(f"  Throughput             : {throughput:.1f} tokens/sec")
    print(f"{'='*55}")
    for i, (p, o) in enumerate(zip(prompts, outputs_text)):
        print(f"\n  Prompt {i+1}: {p[:60]}...")
        print(f"  Output : {o[len(p):len(p)+80]}...")

    return {
        "timings": timings,
        "avg_latency_ms": avg_time * 1000,
        "throughput_tokens_per_sec": throughput,
        "outputs": outputs_text,
    }

print("Running baseline inference...")
base_results = run_inference_benchmark(base_model, tokenizer, PROMPTS, MAX_NEW_TOKENS, label="Base FP16")

Running baseline inference...

  Base FP16 — Inference Benchmark
  Avg latency per prompt : 881.3 ms
  Avg tokens generated   : 50
  Throughput             : 56.7 tokens/sec

  Prompt 1: Artificial intelligence is transforming the world by...
  Output :  creating new ways to solve problems.

The world is changing, and artificial int...

  Prompt 2: The key to a successful machine learning model is...
  Output :  to understand the underlying problem.

The problem is that the model is not a p...

  Prompt 3: Quantum computing will revolutionize cryptography because...
  Output :  it will allow us to store and transmit information in a way that is not possibl...

  Prompt 4: Climate change poses a significant threat to...
  Output :  the planet, and the world is at risk of becoming a carbon-intensive economy.

T...

  Prompt 5: The history of the Roman Empire shows us that...
  Output :  the Roman Empire was a very powerful and powerful force in the world.

The Roma...


In [ ]:
# ✅ CHUNK 5: Quantize with GPTQ (4-bit) using gptqmodel

from gptqmodel import GPTQModel, QuantizeConfig
from datasets import load_dataset
import random

QUANT_OUT_DIR = "./opt-125m-gptq-4bit"

# ─── Calibration data ───────────────────────────────────────────
# Using wikitext-2 (same dataset as the original GPTQ paper)
print("Preparing calibration dataset...")

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
texts = [t for t in dataset["text"] if len(t.strip()) > 100]
random.seed(42)
calib_texts = random.sample(texts, 128)   # 128 samples is standard

def tokenize_calib(texts, tokenizer, seqlen=512):
    samples = []
    for t in texts:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=seqlen)
        if enc["input_ids"].shape[1] >= 32:
            samples.append({"input_ids": enc["input_ids"]})
    return samples

calib_data = tokenize_calib(calib_texts, tokenizer)
print(f"Calibration samples ready: {len(calib_data)}")

# ─── Quantize ───────────────────────────────────────────────────
print("\nStarting GPTQ quantization (4-bit, group_size=128) ...")
print("This takes ~2-4 min on T4 ...\n")

quant_config = QuantizeConfig(
    bits=4,           # 4-bit quantization
    group_size=128,   # standard group size from GPTQ paper
    desc_act=False,   # False = faster, True = slightly better quality
)

quant_model = GPTQModel.load(MODEL_ID, quant_config)

t_start = time.perf_counter()
quant_model.quantize(calib_data, batch_size=4)
t_quant = time.perf_counter() - t_start

print(f"\n✅ Quantization complete in {t_quant:.1f}s ({t_quant/60:.1f} min)")

# Save quantized model
quant_model.save(QUANT_OUT_DIR)
tokenizer.save_pretrained(QUANT_OUT_DIR)
print(f"Saved to: {QUANT_OUT_DIR}")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 6.0.3
Transformers : 5.5.3
Torch        : 2.10.0+cu128
Triton       : 3.6.0


Preparing calibration dataset...


...

INFO  | Capture inputs     | 1     | 0.291  | 0.291 | 0.291   | 0.2%   | cache_inputs:OPTDecoderLayer                    |


INFO  +--------------------+-------+--------+-------+---------+--------+-------------------------------------------------+


INFO  | Process finalize   | 2     | 0.001  | 0.001 | 0.002   | 0.0%   | tp-pre-pad                                      |


INFO  +--------------------+-------+--------+-------+---------+--------+-------------------------------------------------+


Saved to: ./opt-125m-gptq-4bit


In [ ]:
# ✅ CHUNK 6: Load quantized model and run same benchmark

# Free base model from GPU first
del base_model
torch.cuda.empty_cache()

print("Loading quantized model...")
quant_model_loaded = GPTQModel.load(QUANT_OUT_DIR, device="cuda:0")
quant_model_loaded.model.eval()

print("\nRunning quantized model inference...")
quant_results = run_inference_benchmark(
    quant_model_loaded.model,
    tokenizer,
    PROMPTS,
    MAX_NEW_TOKENS,
    label="GPTQ 4-bit"
)

Loading quantized model...
from_quantized: adapter: None


INFO  Loader: Auto dtype (native float16): `torch.float16`                     


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

INFO  HFKernelLinear: loaded CPU gemm_4bit kernel from `kernels-community/quantization-gptq` variant `torch210-cxx11-cpu-x86_64-linux`.


...

INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}



INFO  Kernel: loaded -> `[TritonV2QuantLinear]`                                



Running quantized model inference...

  GPTQ 4-bit — Inference Benchmark
  Avg latency per prompt : 1348.9 ms
  Avg tokens generated   : 50
  Throughput             : 37.1 tokens/sec

  Prompt 1: Artificial intelligence is transforming the world by...
  Output :  creating new ways to communicate with one another.

The world is changing, and ...

  Prompt 2: The key to a successful machine learning model is...
  Output :  to understand the underlying data. This is the key to a successful machine lear...

  Prompt 3: Quantum computing will revolutionize cryptography because...
  Output :  it will allow us to do more with less.

The first quantum computer was invented...

  Prompt 4: Climate change poses a significant threat to...
  Output :  the planet's health and the environment, and it is a major cause of global warm...

  Prompt 5: The history of the Roman Empire shows us that...
  Output :  the Roman Empire was a very powerful and powerful force.

The Roman Empire was ...


In [ ]:
# ✅ CHUNK 7: Comprehensive metrics — Benefits + Degradation

import os
import math

print("\n" + "="*60)
print("       FULL QUANTIZATION METRICS REPORT")
print("="*60)

# ─── 1. Model size on disk ──────────────────────────────────────
def get_dir_size_mb(path):
    total = 0
    for f in os.scandir(path):
        if f.is_file():
            total += f.stat().st_size
    return total / (1024**2)

# Re-save base model temporarily for fair comparison
BASE_SAVE = "./opt-125m-base"
from transformers import AutoModelForCausalLM as AMCLM
base_tmp = AMCLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
base_tmp.save_pretrained(BASE_SAVE)
del base_tmp
torch.cuda.empty_cache()

base_disk_mb  = get_dir_size_mb(BASE_SAVE)
quant_disk_mb = get_dir_size_mb(QUANT_OUT_DIR)
size_reduction = (1 - quant_disk_mb / base_disk_mb) * 100
compression_ratio = base_disk_mb / quant_disk_mb

print(f"\n{'─'*40}")
print(f"  📦 MODEL SIZE (DISK)")
print(f"{'─'*40}")
print(f"  Base FP16          : {base_disk_mb:.1f} MB")
print(f"  GPTQ 4-bit         : {quant_disk_mb:.1f} MB")
print(f"  Size reduction     : {size_reduction:.1f}%")
print(f"  Compression ratio  : {compression_ratio:.2f}x")

# ─── 2. Inference speed ─────────────────────────────────────────
base_lat    = base_results["avg_latency_ms"]
quant_lat   = quant_results["avg_latency_ms"]
base_tput   = base_results["throughput_tokens_per_sec"]
quant_tput  = quant_results["throughput_tokens_per_sec"]
speedup     = base_lat / quant_lat

print(f"\n{'─'*40}")
print(f"  ⚡ INFERENCE SPEED")
print(f"{'─'*40}")
print(f"  Base  avg latency  : {base_lat:.1f} ms/prompt")
print(f"  GPTQ  avg latency  : {quant_lat:.1f} ms/prompt")
print(f"  Speedup            : {speedup:.2f}x  ({'faster ✅' if speedup > 1 else 'slower ⚠️'})")
print(f"  Base  throughput   : {base_tput:.1f} tok/s")
print(f"  GPTQ  throughput   : {quant_tput:.1f} tok/s")

# ─── 3. GPU memory (VRAM) ───────────────────────────────────────
# We measure peak VRAM during inference
def measure_peak_vram(model, tokenizer, prompt):
    torch.cuda.reset_peak_memory_stats()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=50)
    return torch.cuda.max_memory_allocated() / 1e6  # MB

# Reload base model briefly for VRAM measurement
base_tmp2 = AMCLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
base_tmp2.eval()
base_vram = measure_peak_vram(base_tmp2, tokenizer, PROMPTS[0])
del base_tmp2
torch.cuda.empty_cache()

quant_vram = measure_peak_vram(quant_model_loaded.model, tokenizer, PROMPTS[0])
vram_saving = (1 - quant_vram / base_vram) * 100

print(f"\n{'─'*40}")
print(f"  🖥️  GPU VRAM USAGE (peak during inference)")
print(f"{'─'*40}")
print(f"  Base FP16  peak    : {base_vram:.1f} MB")
print(f"  GPTQ 4-bit peak    : {quant_vram:.1f} MB")
print(f"  VRAM saved         : {vram_saving:.1f}%")

# ─── 4. Perplexity (quality degradation metric) ─────────────────
print(f"\n{'─'*40}")
print(f"  📉 PERPLEXITY (lower = better, degradation metric)")
print(f"{'─'*40}")

def compute_perplexity(model, tokenizer, texts, max_len=512, stride=256, n_samples=20):
    model.eval()
    nlls = []
    for text in texts[:n_samples]:
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len)
        input_ids = enc["input_ids"].to("cuda")
        if input_ids.shape[1] < 4:
            continue
        with torch.no_grad():
            out = model(input_ids, labels=input_ids)
        nll = out.loss.item()
        if not math.isnan(nll):
            nlls.append(nll)
    ppl = math.exp(np.mean(nlls)) if nlls else float("inf")
    return ppl

# Use a small validation set
val_texts = [t for t in load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")["text"] if len(t.strip()) > 100]

# Base perplexity
base_tmp3 = AMCLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
base_tmp3.eval()
base_ppl = compute_perplexity(base_tmp3, tokenizer, val_texts)
del base_tmp3
torch.cuda.empty_cache()

# Quantized perplexity
quant_ppl = compute_perplexity(quant_model_loaded.model, tokenizer, val_texts)

ppl_increase = ((quant_ppl - base_ppl) / base_ppl) * 100

print(f"  Base FP16 PPL      : {base_ppl:.2f}")
print(f"  GPTQ 4-bit PPL     : {quant_ppl:.2f}")
print(f"  PPL degradation    : +{ppl_increase:.1f}%  ({'acceptable ✅' if ppl_increase < 15 else 'significant ⚠️'})")

# ─── 5. Output similarity (BLEU-like token overlap) ─────────────
print(f"\n{'─'*40}")
print(f"  🔤 OUTPUT TEXT SIMILARITY")
print(f"{'─'*40}")

def token_overlap(str1, str2):
    t1, t2 = set(str1.lower().split()), set(str2.lower().split())
    if not t1 or not t2: return 0.0
    return len(t1 & t2) / len(t1 | t2)   # Jaccard similarity

overlaps = []
for b_out, q_out in zip(base_results["outputs"], quant_results["outputs"]):
    overlaps.append(token_overlap(b_out, q_out))
avg_overlap = np.mean(overlaps) * 100

print(f"  Avg token overlap  : {avg_overlap:.1f}%")
for i, (score, b, q) in enumerate(zip(overlaps, base_results["outputs"], quant_results["outputs"])):
    print(f"\n  Prompt {i+1} similarity: {score*100:.1f}%")
    print(f"    Base : {b[50:130]}...")
    print(f"    GPTQ : {q[50:130]}...")

# ─── 6. Summary Table ───────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  ✅ SUMMARY")
print(f"{'='*60}")
print(f"  {'Metric':<28} {'Base FP16':>12} {'GPTQ 4-bit':>12}")
print(f"  {'-'*52}")
print(f"  {'Disk size (MB)':<28} {base_disk_mb:>12.1f} {quant_disk_mb:>12.1f}")
print(f"  {'VRAM peak (MB)':<28} {base_vram:>12.1f} {quant_vram:>12.1f}")
print(f"  {'Latency (ms)':<28} {base_lat:>12.1f} {quant_lat:>12.1f}")
print(f"  {'Throughput (tok/s)':<28} {base_tput:>12.1f} {quant_tput:>12.1f}")
print(f"  {'Perplexity':<28} {base_ppl:>12.2f} {quant_ppl:>12.2f}")
print(f"  {'Output similarity':<28} {'—':>12} {avg_overlap:>11.1f}%")
print(f"{'='*60}")
print(f"\n  📌 Benefits  → {size_reduction:.0f}% smaller, {vram_saving:.0f}% less VRAM, {speedup:.2f}x speed")
print(f"  📌 Degradation → PPL +{ppl_increase:.1f}%, output sim {avg_overlap:.0f}%")


       FULL QUANTIZATION METRICS REPORT


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


────────────────────────────────────────
  📦 MODEL SIZE (DISK)
────────────────────────────────────────
  Base FP16          : 238.9 MB
  GPTQ 4-bit         : 122.7 MB
  Size reduction     : 48.6%
  Compression ratio  : 1.95x

────────────────────────────────────────
  ⚡ INFERENCE SPEED
────────────────────────────────────────
  Base  avg latency  : 881.3 ms/prompt
  GPTQ  avg latency  : 1348.9 ms/prompt
  Speedup            : 0.65x  (slower ⚠️)
  Base  throughput   : 56.7 tok/s
  GPTQ  throughput   : 37.1 tok/s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


────────────────────────────────────────
  🖥️  GPU VRAM USAGE (peak during inference)
────────────────────────────────────────
  Base FP16  peak    : 463.8 MB
  GPTQ 4-bit peak    : 212.8 MB
  VRAM saved         : 54.1%

────────────────────────────────────────
  📉 PERPLEXITY (lower = better, degradation metric)
────────────────────────────────────────


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  Base FP16 PPL      : 71.65
  GPTQ 4-bit PPL     : 73.29
  PPL degradation    : +2.3%  (acceptable ✅)

────────────────────────────────────────
  🔤 OUTPUT TEXT SIMILARITY
────────────────────────────────────────
  Avg token overlap  : 46.1%

  Prompt 1 similarity: 52.0%
    Base : by creating new ways to solve problems.

The world is changing, and artificial i...
    GPTQ : by creating new ways to communicate with one another.

The world is changing, an...

  Prompt 2 similarity: 40.7%
    Base : to understand the underlying problem.

The problem is that the model is not a pr...
    GPTQ : to understand the underlying data. This is the key to a successful machine learn...

  Prompt 3 similarity: 31.4%
    Base : because it will allow us to store and transmit information in a way that is not ...
    GPTQ : because it will allow us to do more with less.

The first quantum computer was i...

  Prompt 4 similarity: 29.7%
    Base : lanet, and the world is at risk of becoming a carbon-inte